# Feature Engineering V02

## Objetivo

El objetivo de esta notebook es enriquecer el Match Vector V01 mediante la creación automática de nuevas variables derivadas.

En esta versión va a haber tres familias de features:

- Diferencias entre el equipo local y visitante.
- Ratios entre estadísticas de ambos equipos.
- Sumas de estadísticas de ambos equipos.

Estas nuevas variables deben aportar información comparativa sin incorporar todavía información externa como ranking FIFA, Elo o características individuales de los jugadores.

El resultado será el dataset `match_vector_v02.csv`.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/match_vector_v01.csv")

df.shape

(128, 60)

### Estadísticas

In [6]:
def get_team_statistics(df):
    """
    Devuelve las estadísticas numéricas que existen tanto para home_ como para away_.
    Excluye identificadores, nombres de equipo y variables target/resultados.
    """

    excluded_stats = {
        "match_id",
        "team_name",
        "team",
        "score",
        "year",
    }

    home_stats = {
        col.replace("home_", "")
        for col in df.columns
        if col.startswith("home_")
    }

    away_stats = {
        col.replace("away_", "")
        for col in df.columns
        if col.startswith("away_")
    }

    common_stats = sorted(home_stats.intersection(away_stats))

    valid_stats = [
        stat for stat in common_stats
        if stat not in excluded_stats
        and pd.api.types.is_numeric_dtype(df[f"home_{stat}"])
        and pd.api.types.is_numeric_dtype(df[f"away_{stat}"])
    ]

    return valid_stats

In [7]:
team_stats = get_team_statistics(df)

team_stats

['50/50',
 'Bad Behaviour',
 'Ball Receipt*',
 'Ball Recovery',
 'Block',
 'Carry',
 'Clearance',
 'Dispossessed',
 'Dribble',
 'Dribbled Past',
 'Duel',
 'Error',
 'Foul Committed',
 'Foul Won',
 'Goal Keeper',
 'Interception',
 'Miscontrol',
 'Offside',
 'Own Goal Against',
 'Pass',
 'Player Off',
 'Player On',
 'Pressure',
 'Shield',
 'Shot']

In [8]:
len(team_stats)

25

### Diferencias

In [11]:
def create_difference_features(df):
    """
    Crea automáticamente las diferencias entre
    las estadísticas del equipo local y visitante.
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        df_c[f"diff_{stat}"] = df_c[f"home_{stat}"] - df_c[f"away_{stat}"]

    return df_c

In [15]:
df_v2 = create_difference_features(df)

df_v2

,match_id,match_date,kick_off,year,home_team,away_team,home_score,away_score,competition_stage,home_Ball Receipt*,...,diff_Interception,diff_Miscontrol,diff_Offside,diff_Own Goal Against,diff_Pass,diff_Player Off,diff_Player On,diff_Pressure,diff_Shield,diff_Shot
0,7525,2018-06-14,15:00:00.000,2018,Russia,Saudi Arabia,5,0,Group Stage,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00
1,7578,2018-06-15,12:00:00.000,2018,Egypt,Uruguay,0,1,Group Stage,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00
2,7577,2018-06-15,15:00:00.000,2018,Morocco,Iran,0,1,Group Stage,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00
3,7576,2018-06-15,18:00:00.000,2018,Portugal,Spain,3,3,Group Stage,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00
4,7530,2018-06-16,10:00:00.000,2018,France,Australia,2,1,Group Stage,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,3869354,2022-12-10,21:00:00.000,2022,England,France,1,2,Quarter-finals,642.5,...,-4.000000,-3.250000,-0.250000,0.000000,31.750000,-0.250000,-0.250000,-7.5,-0.750000,-5.75
124,3869519,2022-12-13,21:00:00.000,2022,Argentina,Croatia,3,0,Semi-finals,667.8,...,-3.200000,-1.800000,-0.200000,0.200000,31.000000,0.000000,0.000000,-23.4,-0.400000,2.80
125,3869552,2022-12-14,21:00:00.000,2022,France,Morocco,2,0,Semi-finals,552.6,...,-1.800000,-1.800000,-0.200000,-0.200000,246.200000,-0.200000,-0.200000,-60.4,-0.800000,6.80
126,3869684,2022-12-17,17:00:00.000,2022,Croatia,Morocco,2,1,3rd Place Final,618.0,...,0.666667,-3.000000,-0.166667,-0.166667,271.666667,-0.333333,-0.333333,-17.5,-0.333333,3.00


In [16]:
df_v2.shape

(128, 85)

Corroboro

In [17]:
stat = "Pass"

print(df.loc[127, f"home_{stat}"])
print(df.loc[127, f"away_{stat}"])
print(df_v2.loc[127, f"diff_{stat}"])

653.6666666666666
559.3333333333334
94.33333333333326


### Diferencia relativa

Acá hago una aclaración histórica. Al principio iba a medir simplemente un ratio común. Sin embargo, la diferencia relativa con epsilon me permite tener una llegada más robusta a la misma naturaleza de datos que busco para normalizar las diferencias y que el modelo aprenda bien los patrones. Va a ser interesante que aprenda cuándo se repiten los ratios en los distintos niveles de partido y cómo influyen realmente en los resultados.

In [19]:
def create_relative_difference_features(df, epsilon=1e-6):
    """
    Crea automáticamente diferencias relativas entre
    las estadísticas del equipo local y visitante.

    Fórmula:
    relative_diff_X = (home_X - away_X) / (home_X + away_X + epsilon)

    El resultado queda aproximadamente entre -1 y 1.
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        home_col = f"home_{stat}"
        away_col = f"away_{stat}"
        new_col = f"relative_diff_{stat}"

        df_c[new_col] = (
            df_c[home_col] - df_c[away_col]
        ) / (
            df_c[home_col] + df_c[away_col] + epsilon
        )

    return df_c



In [20]:
df_v2 = create_relative_difference_features(df_v2)

df_v2.shape

(128, 110)

In [21]:
stat = "Pass"

home = df_v2.loc[127, f"home_{stat}"]
away = df_v2.loc[127, f"away_{stat}"]
relative_diff = df_v2.loc[127, f"relative_diff_{stat}"]

print("home:", home)
print("away:", away)
print("relative_diff:", relative_diff)
print("manual:", (home - away) / (home + away + 1e-6))

home: 653.6666666666666
away: 559.3333333333334
relative_diff: 0.077768617688017
manual: 0.077768617688017


### Sumas

Me parece relevante tener esta feature porque da una visión sobre el volumen total. De esta manera queda algo tipo:

- diff_X           : ventaja absoluta
- relative_diff_X  : ventaja proporcional
- sum_X            : volumen total

Este grupo de features me permite romper con la ambigüedad que puede dar cada una por separado.

In [25]:
def create_sum_features(df):
    """
    Crea automáticamente sumas entre las estadísticas
    del equipo local y visitante.

    Fórmula:
    sum_X = home_X + away_X
    """

    df_c = df.copy()

    stats = get_team_statistics(df_c)

    for stat in stats:
        home_col = f"home_{stat}"
        away_col = f"away_{stat}"
        new_col = f"sum_{stat}"

        df_c[new_col] = df_c[home_col] + df_c[away_col]

    return df_c



In [23]:
df_v2 = create_sum_features(df_v2)

df_v2.shape

(128, 135)

In [24]:
stat = "Pass"

home = df_v2.loc[127, f"home_{stat}"]
away = df_v2.loc[127, f"away_{stat}"]
sum_value = df_v2.loc[127, f"sum_{stat}"]

print("home:", home)
print("away:", away)
print("sum:", sum_value)
print("manual:", home + away)

home: 653.6666666666666
away: 559.3333333333334
sum: 1213.0
manual: 1213.0


## Prueba de pipeline

In [26]:
df_v2 = df.copy()

df_v2 = create_difference_features(df_v2)
df_v2 = create_relative_difference_features(df_v2)
df_v2 = create_sum_features(df_v2)

df_v2.shape

(128, 135)

In [27]:
df_v2.isna().sum().sum()

np.int64(0)

In [28]:
np.isinf(df_v2.select_dtypes(include=[np.number])).sum().sum()

np.int64(0)

In [29]:
df_v2.dtypes.value_counts()

float64    125
int64        5
str          5
Name: count, dtype: int64

In [30]:
new_cols = [col for col in df_v2.columns if col not in df.columns]

len(new_cols), new_cols[:20]

(75,
 ['diff_50/50',
  'diff_Bad Behaviour',
  'diff_Ball Receipt*',
  'diff_Ball Recovery',
  'diff_Block',
  'diff_Carry',
  'diff_Clearance',
  'diff_Dispossessed',
  'diff_Dribble',
  'diff_Dribbled Past',
  'diff_Duel',
  'diff_Error',
  'diff_Foul Committed',
  'diff_Foul Won',
  'diff_Goal Keeper',
  'diff_Interception',
  'diff_Miscontrol',
  'diff_Offside',
  'diff_Own Goal Against',
  'diff_Pass'])

## Guardado de .csv

In [31]:
df_v2.to_csv("../data/processed/match_vector_v02.csv", index=False)

In [32]:
test = pd.read_csv("../data/processed/match_vector_v02.csv")

test.shape

(128, 135)